# **COMP1110 Group Project: Restaurant Queue Management**

# **1. Data Model**

In [1]:
import random
import csv
import math

In [23]:
import json # read "*.json" file
import pandas as pd # data processing and analysis
import numpy as np # computing values
import matplotlib.pyplot as plt # data visualisation
from datetime import datetime, timedelta # time function

Attributes needed:
* `group_id`：唯一编号
* `customer_type`：walkin 或 reservation
* `reserved_time`：预约时间
* `actual_arrival_time`：实际到达时间
* `group_size`：人数
* `dining_duration`：用餐时长
* `no_show`：是否爽约
* `cancelled`：是否取消

可考虑这几个方面：
* 谁来了
* 几个人
* 什么时候来
* 吃多久
* 是 walk-in 还是 reservation
* reservation 的 scheduled time 是多少
* 有没有 no-show / cancellation

## Scratch Data 1

In [6]:
df = pd.read_csv('data.csv')
df['reserved_time'] = pd.to_datetime(df['reserved_time'], format='%H:%M', errors='coerce')
df['actual_arrival_time'] = pd.to_datetime(df['actual_arrival_time'], format='%H:%M', errors='coerce')
df


,group_id,customer_type,reserved_time,actual_arrival_time,group_size,dining_duration,no_show,cancelled
0,G001,walkin,NaN,07:05,1,22,0,0
1,G002,reservation,07:15,07:12,2,34,0,0
2,G003,walkin,NaN,07:28,2,30,0,0
3,G004,reservation,07:45,07:49,3,40,0,0
4,G005,walkin,NaN,08:03,1,25,0,0
...,...,...,...,...,...,...,...,...
63,G064,reservation,20:30,NaN,5,0,0,1
64,G065,walkin,NaN,20:44,3,67,0,0
65,G066,reservation,21:00,21:06,2,58,0,0
66,G067,walkin,NaN,21:18,2,42,0,0


## Scratch Data 2

In [ ]:
data = {
    'arrival_time': ['11:30', '12:15', '12:30', '13:00', '13:20'],
    'group_size': [2, 4, 3, 1, 5],
    'dining_duration': [45, 60, 50, 30, 75]
}
df['arrival_time'] = pd.to_datetime(df['arrival_time'], format='%H:%M')
df = pd.DataFrame(data)
print(df)

  arrival_time  group_size  dining_duration
0        11:30           2               45
1        12:15           4               60
2        12:30           3               50
3        13:00           1               30
4        13:20           5               75


Setting a dataframe as shown above, and add a column `finish_time` to compute the finish time of the customers in restaurants.

In [27]:
df['arrival_time'] = pd.to_datetime(df['arrival_time'], format='%H:%M')
df['dining_timedelta'] = pd.to_timedelta(df['dining_duration'], unit='m')
df['finish_time'] = df['arrival_time'] + df['dining_timedelta']
df['finish_time'] = df['finish_time'].dt.strftime('%H:%M')
df['arrival_time'] = df['arrival_time'].dt.strftime('%H:%M')
print(df[['arrival_time', 'group_size', 'dining_duration', 'finish_time']])

  arrival_time  group_size  dining_duration finish_time
0        11:30           2               45       12:15
1        12:15           4               60       13:15
2        12:30           3               50       13:20
3        13:00           1               30       13:30
4        13:20           5               75       14:35


In [ ]:
group_size = []
arrival_time = []
dining_duration = []

## **Formal Data**

In [113]:
df = pd.read_csv('data.csv')
df

,group_id,customer_type,reserved_time,actual_arrival_time,group_size,dining_duration,no_show,cancelled
0,G001,walkin,NaN,07:05,1,22,0,0
1,G002,reservation,07:15,07:12,2,34,0,0
2,G003,walkin,NaN,07:28,2,30,0,0
3,G004,reservation,07:45,07:49,3,40,0,0
4,G005,walkin,NaN,08:03,1,25,0,0
...,...,...,...,...,...,...,...,...
63,G064,reservation,20:30,NaN,5,0,0,1
64,G065,walkin,NaN,20:44,3,67,0,0
65,G066,reservation,21:00,21:06,2,58,0,0
66,G067,walkin,NaN,21:18,2,42,0,0


In [114]:
df['reserved_time'] = pd.to_datetime(df['reserved_time'], format='%H:%M', errors='coerce')
df['actual_arrival_time'] = pd.to_datetime(df['actual_arrival_time'], format='%H:%M', errors='coerce')

# **2. Data Processing**

### 2.1 Calculate finishing time, `actual_finish_time`

In [115]:
df['actual_finish_time'] = df['actual_arrival_time'] + pd.to_timedelta(df['dining_duration'], unit='m')
df['actual_finish_time'] = df['actual_finish_time'].dt.strftime('%H:%M')

df['reserved_time'] = df['reserved_time'].dt.strftime('%H:%M')
df['actual_arrival_time'] = df['actual_arrival_time'].dt.strftime('%H:%M')

df

,group_id,customer_type,reserved_time,actual_arrival_time,group_size,dining_duration,no_show,cancelled,actual_finish_time
0,G001,walkin,NaN,07:05,1,22,0,0,07:27
1,G002,reservation,07:15,07:12,2,34,0,0,07:46
2,G003,walkin,NaN,07:28,2,30,0,0,07:58
3,G004,reservation,07:45,07:49,3,40,0,0,08:29
4,G005,walkin,NaN,08:03,1,25,0,0,08:28
...,...,...,...,...,...,...,...,...,...
63,G064,reservation,20:30,NaN,5,0,0,1,NaN
64,G065,walkin,NaN,20:44,3,67,0,0,21:51
65,G066,reservation,21:00,21:06,2,58,0,0,22:04
66,G067,walkin,NaN,21:18,2,42,0,0,22:00


### 2.2 Calculate Waiting Time

In [123]:
df['actual_finish_time_0'] = pd.to_datetime(df['actual_finish_time'], format='%H:%M', errors='coerce')
df['reserved_time_0'] = pd.to_datetime(df['reserved_time'], format='%H:%M', errors='coerce')

df['prev_actual_finish'] = df['actual_finish_time_0'].shift(1)
df['waiting_time'] = (df['prev_actual_finish'] - df['reserved_time_0']).dt.total_seconds() / 60

df.drop(columns=['prev_actual_finish', 'reserved_time_0', 'actual_finish_time_0'], inplace=True)

df

,group_id,customer_type,reserved_time,actual_arrival_time,group_size,dining_duration,no_show,cancelled,actual_finish_time,waiting_time
0,G001,walkin,NaN,07:05,1,22,0,0,07:27,NaN
1,G002,reservation,07:15,07:12,2,34,0,0,07:46,12.0
2,G003,walkin,NaN,07:28,2,30,0,0,07:58,NaN
3,G004,reservation,07:45,07:49,3,40,0,0,08:29,13.0
4,G005,walkin,NaN,08:03,1,25,0,0,08:28,NaN
...,...,...,...,...,...,...,...,...,...,...
63,G064,reservation,20:30,NaN,5,0,0,1,NaN,36.0
64,G065,walkin,NaN,20:44,3,67,0,0,21:51,NaN
65,G066,reservation,21:00,21:06,2,58,0,0,22:04,51.0
66,G067,walkin,NaN,21:18,2,42,0,0,22:00,NaN


In [124]:
df.to_csv('newer_set.csv', index=False)